# 1. What you'll learn

        - measure variance reduction over bootstrap resamples
- use out-of-bag evaluation
- compare impurity and permutation importance

        **The one question this notebook answers:** Can unstable trees become stable when their errors are diverse and averaged?

# 2. Setup

This lesson keeps bagging and random forests small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score,roc_auc_score

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

The same 569-row breast-cancer data expose both nonlinear prediction and ambiguity among correlated size measurements.

**Small example:** Five tree probabilities [0.1,0.8,0.6,0.7,0.3] average to 0.5. Different errors cancel; five identical trees add no stability.

In [ ]:
b=load_breast_cancer(as_frame=True);X=b.data;y=(b.target==0).astype(int);print("Shape:",X.shape,"malignant:",round(y.mean(),3));print(X.iloc[:3,:5].round(2))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
rng=np.random.default_rng(SEED);boot=rng.choice(len(X),len(X),replace=True);corr=X[["mean radius","mean perimeter","mean area"]].corr();fig,axes=plt.subplots(1,3,figsize=(14,3.7));y.value_counts().plot.bar(ax=axes[0],color="#176b66");axes[0].set_title("Outcome counts")
sns.heatmap(corr,annot=True,vmin=0,vmax=1,cmap="mako",ax=axes[1]);axes[1].set_title("Redundant size features")
u=len(np.unique(boot));axes[2].bar(["unique bootstrap","left OOB"],[u,len(X)-u],color=["#176b66","#d38b45"]);axes[2].set_title("One n-row sample");plt.tight_layout();plt.show()

**Takeaway:** Class balance motivates balanced metrics.

**Takeaway:** Correlated predictors can substitute for one another.

**Takeaway:** Bootstrap omission creates both tree diversity and OOB evaluation rows.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED);forest=RandomForestClassifier(n_estimators=180,max_features="sqrt",min_samples_leaf=2,oob_score=True,n_jobs=1,random_state=SEED,class_weight="balanced");print("Train/test:",X_train.shape,X_test.shape)

# 6. Train

        Training turns the assumptions behind bagging and random forests into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Bootstrap training rows for every tree.
2. Offer a random feature subset at each node.
3. Grow diverse trees on perturbed data.
4. Average probabilities and score each training row with trees where it was OOB.

In [ ]:
forest.fit(X_train,y_train);prob=forest.predict_proba(X_test)[:,1];print(f"OOB={forest.oob_score_:.3f}; test balanced={balanced_accuracy_score(y_test,prob>=.5):.3f}; AUC={roc_auc_score(y_test,prob):.3f}")

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
rng=np.random.default_rng(SEED);tp=[];fp=[]
for s in range(18):
 idx=rng.choice(len(X_train),len(X_train),replace=True);tp.append(DecisionTreeClassifier(random_state=s).fit(X_train.iloc[idx],y_train.iloc[idx]).predict_proba(X_test)[:,1]);fp.append(RandomForestClassifier(n_estimators=45,max_features="sqrt",min_samples_leaf=2,n_jobs=1,random_state=s).fit(X_train.iloc[idx],y_train.iloc[idx]).predict_proba(X_test)[:,1])
tv=np.var(tp,axis=0).mean();fv=np.var(fp,axis=0).mean();perm=permutation_importance(forest,X_test,y_test,n_repeats=5,random_state=SEED,n_jobs=1);a=np.argsort(forest.feature_importances_)[-6:];bidx=np.argsort(perm.importances_mean)[-6:]
fig,axes=plt.subplots(1,3,figsize=(15,3.9));axes[0].bar(["tree","forest"],[tv,fv],color=["#d38b45","#176b66"]);axes[0].set_title("Resample prediction variance");axes[1].barh(X.columns[a],forest.feature_importances_[a]);axes[1].set_title("Impurity importance");axes[2].barh(X.columns[bidx],perm.importances_mean[bidx]);axes[2].set_title("Permutation importance");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"Variance reduction={1-fv/tv:.1%}; balanced baseline=.500")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Averaging only helps when resampling and random feature choices make tree errors imperfectly correlated. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
identical=[DecisionTreeClassifier(random_state=s).fit(X_train,y_train).predict_proba(X_test)[:,1] for s in range(20)];diverse=[RandomForestClassifier(n_estimators=1,max_features="sqrt",random_state=s).fit(X_train,y_train).predict_proba(X_test)[:,1] for s in range(20)]
print(f"Identical-tree variance={np.var(identical,axis=0).mean():.6f}; randomized-tree variance={np.var(diverse,axis=0).mean():.4f}")

**Use this when…** tabular nonlinearities matter and stability is worth less single-tree transparency.

        **Don't use this when…** you need extrapolation, a compact formula, or causal feature importance.

        ## Try this

        1. Change max_features and measure tree correlation.
2. Plot OOB error from 5 to 300 trees.
3. Duplicate a strong feature and watch importance split.